# RAG 파이프라인 구축 (Retriever 세팅)

챗봇이 답변을 생성할 때, 외부 지식을 참조할 수 있도록 RAG 파이프라인을 구축합니다.

LangChain이나 LangGraph 같은 프레임워크를 활용하면 이 과정을 매우 정교하게 파이프라인화할 수 있습니다.

In [1]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

C:\Users\muckj\miniconda3\envs\pandas_2\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# 1. 문서 로드 및 스플릿
loader = TextLoader("store_manual.txt")
docs = loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
splits = text_splitter.split_documents(docs)

# 2. 한국어 임베딩 모델 적용 및 벡터 DB(Chroma) 구축
embeddings = HuggingFaceEmbeddings(model_name="jhgan/ko-sroberta-multitask")
vectorstore = Chroma.from_documents(
    documents=splits, 
    embedding=embeddings,
    collection_metadata={"hnsw:space": "cosine"} # 코사인 유사도로 검색 공간을 변경합니다.
)

# 3. Retriever (검색기) 생성
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})